In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files (they are in current working directory)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

def fmt_ts_epoch_to_local(ts):
    if pd.isna(ts):
        return None
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

def summarize_df(df, name_col):
    total_rows = int(df.shape[0])
    if 'timestamp' in df.columns and total_rows>0:
        min_ts = int(df['timestamp'].min())
        max_ts = int(df['timestamp'].max())
        min_ts_local = fmt_ts_epoch_to_local(min_ts)
        max_ts_local = fmt_ts_epoch_to_local(max_ts)
    else:
        min_ts = None
        max_ts = None
        min_ts_local = None
        max_ts_local = None

    unique_cmdb = sorted(df['cmdb_id'].dropna().unique().tolist()) if 'cmdb_id' in df.columns else []
    unique_cmdb_series = pd.Series(unique_cmdb, name='unique_cmdb_ids')

    if name_col in df.columns:
        top_names = df[name_col].fillna('(null)').value_counts().head(20).reset_index()
        top_names.columns = [name_col, 'count']
    else:
        top_names = pd.DataFrame(columns=[name_col, 'count'])

    summary = pd.Series({
        'total_rows': total_rows,
        'min_timestamp_epoch': min_ts,
        'min_timestamp_local(Asia/Shanghai)': min_ts_local,
        'max_timestamp_epoch': max_ts,
        'max_timestamp_local(Asia/Shanghai)': max_ts_local,
        'unique_cmdb_count': len(unique_cmdb),
    })

    return summary, unique_cmdb_series, top_names

# Summarize each file
summary_metric, unique_metric_cmdb, top_metric_kpis = summarize_df(df_metric, 'kpi_name')
summary_trace, unique_trace_cmdb, top_trace_names = summarize_df(df_trace, 'trace_name')
summary_log, unique_log_cmdb, top_log_names = summarize_df(df_log, 'log_name')
summary_error, unique_error_cmdb, top_error_messages = summarize_df(df_error, 'message')

# Display compact results (variables separated by commas)
summary_metric, unique_metric_cmdb, top_metric_kpis, summary_trace, unique_trace_cmdb, top_trace_names, summary_log, unique_log_cmdb, top_log_names, summary_error, unique_error_cmdb, top_error_messages

```
Out[1]:
```
import pandas as pd

# Compose a concise plain-English summary from existing analysis variables
metric_rows = int(summary_metric['total_rows'])
metric_min = summary_metric['min_timestamp_local(Asia/Shanghai)']
metric_max = summary_metric['max_timestamp_local(Asia/Shanghai)']
metric_cmdb_list = ', '.join(unique_metric_cmdb.tolist())
metric_cmdb_count = int(summary_metric['unique_cmdb_count'])
metric_top = '; '.join([f"{r['kpi_name']} ({int(r['count'])})" for _, r in top_metric_kpis.iterrows()])

trace_rows = int(summary_trace['total_rows'])
trace_min = summary_trace['min_timestamp_local(Asia/Shanghai)']
trace_max = summary_trace['max_timestamp_local(Asia/Shanghai)']
trace_cmdb_list = ', '.join(unique_trace_cmdb.tolist())
trace_cmdb_count = int(summary_trace['unique_cmdb_count'])
trace_top = '; '.join([f"{r['trace_name']} ({int(r['count'])})" for _, r in top_trace_names.iterrows()])

log_rows = int(summary_log['total_rows'])
log_min = summary_log['min_timestamp_local(Asia/Shanghai)']
log_max = summary_log['max_timestamp_local(Asia/Shanghai)']
log_cmdb_list = ', '.join(unique_log_cmdb.tolist())
log_cmdb_count = int(summary_log['unique_cmdb_count'])
log_top = '; '.join([f"{r['log_name']} ({int(r['count'])})" for _, r in top_log_names.iterrows()])

error_rows = int(summary_error['total_rows'])
error_summary = "no rows" if error_rows == 0 else f"{int(summary_error['total_rows'])} rows"
error_cmdb_count = int(summary_error['unique_cmdb_count'])
error_cmdb_list = ', '.join(unique_error_cmdb.tolist()) if len(unique_error_cmdb) > 0 else '(none)'
error_top = '; '.join([f"{r['message']} ({int(r['count'])})" for _, r in top_error_messages.iterrows()]) if not top_error_messages.empty else '(none)'

summary = (
    f"metric.csv: {metric_rows} rows; time range {metric_min} to {metric_max} (Asia/Shanghai); "
    f"{metric_cmdb_count} unique cmdb_id(s): {metric_cmdb_list}. Top KPI names (top up to 20): {metric_top}\n\n"
    f"trace.csv: {trace_rows} rows; time range {trace_min} to {trace_max} (Asia/Shanghai); "
    f"{trace_cmdb_count} unique cmdb_id(s): {trace_cmdb_list}. Top trace names (top up to 20): {trace_top}\n\n"
    f"log.csv: {log_rows} rows; time range {log_min} to {log_max} (Asia/Shanghai); "
    f"{log_cmdb_count} unique cmdb_id(s): {log_cmdb_list}. Top log names: {log_top}\n\n"
    f"error_logs.csv: {error_summary}; {error_cmdb_count} unique cmdb_id(s): {error_cmdb_list}. Top messages: {error_top}"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(total_rows                                              29355
min_timestamp_epoch                                1616634000
min_timestamp_local(Asia/Shanghai)    2021-03-25 09:00:00 CST
max_timestamp_epoch                                1616635800
max_timestamp_local(Asia/Shanghai)    2021-03-25 09:30:00 CST
unique_cmdb_count                                          26
dtype: object, 0         IG01
1         IG02
2         MG01
3         MG02
4      Mysql01
        ...   
21    Tomcat03
22    Tomcat04
23    apache01
24    apache02
25    dockerA2
Name: unique_cmdb_ids, Length: 26, dtype: object,                                              kpi_name  count
0      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    349
1       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    347
2          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    347
3                  OSLinux-OSLinux_ZABBIX_Host_Uptime    346
4        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    344
5                         OSLinux-CPU_CPU_CPUidleutil    343
6                             OSLinux-CPU_CPU_CPULoad    343
7   OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...    341
8                          OSLinux-CPU_CPU_CPUSysTime    341
9                         OSLinux-CPU_CPU_CPUUserTime    341
10                             OSLinux-CPU_CPU_CPUWio    341
11                                            app.mrt    333
12                                             app.sr    333
13                                             app.rr    333
14                                            app.cnt    333
15       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc    295
16                         OSLinux-CPU_CPU_CPUCpuUtil    282
17       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    280
18              OSLinux-OSLinux_MEMORY_MEMORY_UserMem    267
19          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    261, total_rows                                               7203
min_timestamp_epoch                                1616634000
min_timestamp_local(Asia/Shanghai)    2021-03-25 09:00:00 CST
max_timestamp_epoch                                1616635800
max_timestamp_local(Asia/Shanghai)    2021-03-25 09:30:00 CST
unique_cmdb_count                                          12
dtype: object, 0         IG01
1         IG02
2         MG01
3         MG02
4     Tomcat01
5     Tomcat02
6     Tomcat03
7     Tomcat04
8     dockerA1
9     dockerA2
10    dockerB1
11    dockerB2
Name: unique_cmdb_ids, dtype: object,                        trace_name  count
0        trace.self.duration_mean    372
1         trace.self.duration_p95    372
2            trace.self.row_count    372
3         trace.to_MG02.row_count    247
4     trace.to_MG02.duration_mean    247
5      trace.to_MG02.duration_p95    247
6         trace.to_MG01.row_count    245
7     trace.to_MG01.duration_mean    245
8      trace.to_MG01.duration_p95    245
9    trace.from_IG02.duration_p95    124
10      trace.from_MG02.row_count    124
11  trace.from_IG01.duration_mean    124
12      trace.from_IG02.row_count    124
13   trace.from_IG01.duration_p95    124
14      trace.from_IG01.row_count    124
15  trace.from_MG02.duration_mean    124
16   trace.from_MG02.duration_p95    124
17  trace.from_IG02.duration_mean    124
18  trace.from_MG01.duration_mean    123
19   trace.from_MG01.duration_p95    123, total_rows                                                463
min_timestamp_epoch                                1616634000
min_timestamp_local(Asia/Shanghai)    2021-03-25 09:00:00 CST
max_timestamp_epoch                                1616635800
max_timestamp_local(Asia/Shanghai)    2021-03-25 09:30:00 CST
unique_cmdb_count                                           6
dtype: object, 0    Tomcat01
1    Tomcat02
2    Tomcat03
3    Tomcat04
4    apache01
5    apache02
Name: unique_cmdb_ids, dtype: object,            log_name  count
0   log.error_count    186
1     log.row_count    186
2  log.gc_pause_max     91, total_rows                            0.0
min_timestamp_epoch                   NaN
min_timestamp_local(Asia/Shanghai)    NaN
max_timestamp_epoch                   NaN
max_timestamp_local(Asia/Shanghai)    NaN
unique_cmdb_count                     0.0
dtype: float64, Series([], Name: unique_cmdb_ids, dtype: object), Empty DataFrame
Columns: [message, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure proper dtypes
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# 1) Aggregate full metric series (global statistics BEFORE any filtering)
agg_funcs = {
    'value': ['count', lambda x: x.quantile(0.95), lambda x: x.quantile(0.90), lambda x: x.quantile(0.05), 'min', 'max']
}
df_series_stats = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(agg_funcs)
)
# Flatten columns
df_series_stats.columns = [
    'cmdb_id', 'kpi_name',
    'series_count', 'global_p95', 'global_p90', 'global_p5', 'series_min', 'series_max'
]
# Convert counts to int
df_series_stats['series_count'] = df_series_stats['series_count'].astype(int)

# 2) Filter to incident window and compute window metrics and anomalies (using global_p95 from df_series_stats)
start_ts = 1616634000
end_ts = 1616635800
df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Join global thresholds into windowed points
df_window_with_thresholds = pd.merge(
    df_window,
    df_series_stats[['cmdb_id', 'kpi_name', 'global_p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Mark anomalies where value > global_p95 (global_p95 computed on full series)
df_window_with_thresholds['is_anom_p95'] = df_window_with_thresholds['value'] > df_window_with_thresholds['global_p95']

# Group windowed results per cmdb_id + kpi_name
grp = df_window_with_thresholds.groupby(['cmdb_id', 'kpi_name'], as_index=False)

df_window_stats = grp.agg(
    window_count = ('value', 'count'),
    anomaly_count_p95 = ('is_anom_p95', 'sum'),
    max_in_window = ('value', 'max'),
    mean_in_window = ('value', 'mean')
)

# Compute earliest anomaly timestamp per group (where is_anom_p95 is True)
earliest_anom = (
    df_window_with_thresholds[df_window_with_thresholds['is_anom_p95']]
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(earliest_anom_ts_p95=('timestamp', 'min'))
)
# Merge earliest timestamps (may have missing groups -> no anomalies)
df_window_stats = pd.merge(df_window_stats, earliest_anom, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Merge series-level stats with window-level stats to form report
df_report = pd.merge(
    df_series_stats,
    df_window_stats,
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Fill NaNs for groups that had no points in window
df_report['window_count'] = df_report['window_count'].fillna(0).astype(int)
df_report['anomaly_count_p95'] = df_report['anomaly_count_p95'].fillna(0).astype(int)
df_report['max_in_window'] = df_report['max_in_window']  # may remain NaN
df_report['mean_in_window'] = df_report['mean_in_window']  # may remain NaN

# Convert earliest_anom_ts_p95 epoch -> localized string; keep nulls as None
def epoch_to_local_str(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

df_report['earliest_anom_ts_p95'] = df_report['earliest_anom_ts_p95'].apply(epoch_to_local_str)

# Keep only requested columns and sort by anomaly_count_p95 desc, then by series_count desc
report_cols = [
    'cmdb_id', 'kpi_name',
    'series_count', 'global_p95', 'global_p90', 'global_p5', 'series_min', 'series_max',
    'window_count', 'anomaly_count_p95', 'earliest_anom_ts_p95', 'max_in_window', 'mean_in_window'
]
df_report = df_report[report_cols]
df_report_sorted = df_report.sort_values(['anomaly_count_p95', 'window_count', 'series_count'], ascending=[False, False, False])

df_report_top20 = df_report_sorted.head(20).reset_index(drop=True)

# 4) Small list of unique cmdb_id values that have anomaly_count_p95 > 0 with count of distinct anomalous KPIs per cmdb_id (top 20)
df_anom = df_report[df_report['anomaly_count_p95'] > 0].copy()
df_anom_summary = (
    df_anom.groupby('cmdb_id', as_index=False)
    .agg(anomalous_kpi_count=('kpi_name', 'nunique'),
         total_anomaly_points=('anomaly_count_p95', 'sum'))
    .sort_values(['anomalous_kpi_count', 'total_anomaly_points'], ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# Compact display as required: do not print full series, only top tables
df_report_top20, df_anom_summary

```
Out[2]:
```
import pandas as pd

# Use existing kernel variables: df_report_top20, df_anom_summary, df_report (from previous steps)
# Build a compact plain-English summary string.

# Safety checks
for var in ['df_report_top20', 'df_anom_summary', 'df_report']:
    if var not in globals():
        raise NameError(f"Required variable '{var}' not found in kernel state.")

# Aggregate counts for overall anomalous groups
df_anom_all = df_report[df_report['anomaly_count_p95'] > 0].copy()
num_anomalous_groups = int(df_anom_all.shape[0])
total_anomaly_points = int(df_anom_all['anomaly_count_p95'].sum())

# Prepare a few example anomalous component-KPI entries (up to 5) from the top-20 report
examples = []
for _, row in df_report_top20.head(5).iterrows():
    examples.append(
        f"{row['cmdb_id']} | {row['kpi_name']} — anomalies:{int(row['anomaly_count_p95'])}, "
        f"earliest:{row['earliest_anom_ts_p95']}, max_in_window:{row['max_in_window']:.6g}, mean_in_window:{row['mean_in_window']:.6g}"
    )

# Prepare top cmdb_id summary (from df_anom_summary) — show up to top 10
top_components = []
for _, r in df_anom_summary.head(10).iterrows():
    top_components.append(f"{r['cmdb_id']}: {int(r['anomalous_kpi_count'])} anomalous KPIs, {int(r['total_anomaly_points'])} anomaly points")

summary = (
    "Summary of metric.csv anomaly analysis (using global p95 thresholds computed on the full series):\n\n"
    f"- Number of component-KPI groups with anomalies in the incident window: {num_anomalous_groups}\n"
    f"- Total anomaly points (value > global_p95) observed in the window: {total_anomaly_points}\n\n"
    "- Examples (up to 5) from the top results:\n  " + "\n  ".join(examples) + "\n\n"
    "- Top components by distinct anomalous KPIs (top 10):\n  " + "\n  ".join(top_components) + "\n\n"
    "Notes: per-group global_p95/p90/p5 and full-series counts were computed from the entire metric.csv before filtering; "
    "window metrics (counts, earliest anomaly timestamp, max, mean) are computed for timestamps between 1616634000 and 1616635800 (inclusive)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                      kpi_name  series_count    global_p95    global_p90     global_p5    series_min    series_max  window_count  anomaly_count_p95     earliest_anom_ts_p95  max_in_window  mean_in_window
0           IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryUsed            31  8.233135e+08  8.031985e+08  1.122603e+08  7.251458e+07  8.565236e+08            31                  2  2021-03-25 09:05:00 CST   8.565236e+08    4.687445e+08
1           IG02     JVM-Memory_7778_JVM_Memory_HeapMemoryUsed            31  7.952730e+08  7.357539e+08  6.671062e+07  5.843179e+07  8.785233e+08            31                  2  2021-03-25 09:08:00 CST   8.785233e+08    4.380082e+08
2           MG01     JVM-Memory_7779_JVM_Memory_HeapMemoryUsed            31  9.880494e+08  9.180374e+08  2.634802e+08  1.949358e+08  1.008732e+09            31                  2  2021-03-25 09:07:00 CST   1.008732e+09    6.470318e+08
3           MG01      JVM-Threads_7779_JVM_ThreadCount_Threads            31  7.550000e+01  7.500000e+01  6.900000e+01  6.800000e+01  7.700000e+01            31                  2  2021-03-25 09:18:00 CST   7.700000e+01    7.254839e+01
4           MG01            OSLinux-OSLinux_ZABBIX_Host_Uptime            31  7.835434e+06  7.835343e+06  7.833808e+06  7.833717e+06  7.835523e+06            31                  2  2021-03-25 09:29:00 CST   7.835523e+06    7.834620e+06
5           MG02     JVM-Memory_7779_JVM_Memory_HeapMemoryUsed            31  9.549292e+08  9.448733e+08  2.063339e+08  1.594243e+08  9.698863e+08            31                  2  2021-03-25 09:06:00 CST   9.698863e+08    5.820550e+08
6           MG02  OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31  5.000000e-01  0.000000e+00  0.000000e+00  0.000000e+00  1.000000e+00            31                  2  2021-03-25 09:06:00 CST   1.000000e+00    6.451613e-02
7        Redis02                       OSLinux-CPU_CPU_CPULoad            31  2.400000e-01  2.300000e-01  3.000000e-02  1.000000e-02  2.600000e-01            31                  2  2021-03-25 09:00:00 CST   2.600000e-01    1.306452e-01
8        Redis02                    OSLinux-CPU_CPU_CPUSysTime            31  1.584979e+00  1.502842e+00  1.025635e+00  9.126690e-01  1.621100e+00            31                  2  2021-03-25 09:25:00 CST   1.621100e+00    1.282129e+00
9        Redis02                   OSLinux-CPU_CPU_CPUUserTime            31  5.366110e-01  4.947380e-01  3.649545e-01  3.265510e-01  6.037740e-01            31                  2  2021-03-25 09:01:00 CST   6.037740e-01    4.438374e-01
10       Redis02                        OSLinux-CPU_CPU_CPUWio            31  1.655750e-01  1.259000e-01  4.200000e-03  4.199000e-03  5.841000e-01            31                  2  2021-03-25 09:01:00 CST   5.841000e-01    4.793806e-02
11       Redis02                   OSLinux-CPU_CPU_CPUidleutil            31  9.825280e+01  9.811535e+01  9.734010e+01  9.725600e+01  9.850121e+01            31                  2  2021-03-25 09:00:00 CST   9.850121e+01    9.787783e+01
12       Redis02            OSLinux-OSLinux_ZABBIX_Host_Uptime            31  6.285698e+06  6.285607e+06  6.284051e+06  6.283958e+06  6.285789e+06            31                  2  2021-03-25 09:29:00 CST   6.285789e+06    6.284875e+06
13  ServiceTest1                                       app.cnt            31  3.435000e+02  2.760000e+02  3.200000e+01  2.200000e+01  4.670000e+02            31                  2  2021-03-25 09:19:00 CST   4.670000e+02    1.801935e+02
14  ServiceTest1                                       app.mrt            31  2.581500e-01  2.240000e-01  1.273100e-01  9.253000e-02  4.412700e-01            31                  2  2021-03-25 09:24:00 CST   4.412700e-01    1.763623e-01
15  ServiceTest2                                       app.cnt            31  2.335000e+02  2.060000e+02  2.800000e+01  2.100000e+01  2.860000e+02            31                  2  2021-03-25 09:19:00 CST   2.860000e+02    1.358065e+02
16  ServiceTest2                                       app.mrt            31  3.374450e-01  3.233100e-01  1.621650e-01  1.420500e-01  3.845500e-01            31                  2  2021-03-25 09:15:00 CST   3.845500e-01    2.576594e-01
17  ServiceTest4                                       app.cnt            31  2.410000e+02  2.210000e+02  3.400000e+01  2.800000e+01  2.980000e+02            31                  2  2021-03-25 09:24:00 CST   2.980000e+02    1.179355e+02
18  ServiceTest4                                       app.mrt            31  2.835975e+00  2.697190e+00  1.513000e+00  1.358510e+00  3.078270e+00            31                  2  2021-03-25 09:13:00 CST   3.078270e+00    2.064984e+00
19  ServiceTest5                                       app.cnt            31  3.025000e+02  2.530000e+02  2.750000e+01  2.000000e+01  3.760000e+02            31                  2  2021-03-25 09:02:00 CST   3.760000e+02    1.418710e+02,           cmdb_id  anomalous_kpi_count  total_anomaly_points
0         Mysql02                  102                   151
1         Mysql01                   81                   118
2        Tomcat01                   50                    59
3            MG01                   50                    58
4        Tomcat04                   45                    54
5            MG02                   44                    52
6        Tomcat02                   42                    51
7        Tomcat03                   40                    49
8            IG01                   38                    40
9            IG02                   37                    38
10        Redis01                   33                    34
11        Redis02                   30                    36
12       apache01                   26                    26
13       apache02                   26                    26
14       dockerA2                    4                     4
15   ServiceTest1                    2                     4
16  ServiceTest10                    2                     4
17  ServiceTest11                    2                     4
18   ServiceTest2                    2                     4
19   ServiceTest3                    2                     4)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing variables if present, otherwise load files
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure correct dtypes
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Compute global thresholds (full series) grouped by cmdb_id and trace_name
trace_agg = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        series_count=('value', 'count'),
        global_p95=('value', lambda x: x.quantile(0.95) if x.notna().any() else float('nan')),
        global_p5 =('value', lambda x: x.quantile(0.05) if x.notna().any() else float('nan'))
    )
)
# Ensure integer series_count
trace_agg['series_count'] = trace_agg['series_count'].astype(int)

# 2) Filter to incident window and compute window metrics and anomalies
start_ts = 1616634000
end_ts = 1616635800
df_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Merge global thresholds into windowed points
df_trace_win_with_thresh = pd.merge(
    df_trace_window,
    trace_agg[['cmdb_id', 'trace_name', 'global_p95', 'global_p5']],
    on=['cmdb_id', 'trace_name'],
    how='left'
)

# Mark anomaly types:
# - duration anomalies: trace_name contains 'duration' and value > global_p95
# - row_count anomalies: trace_name contains 'row_count' and value <= global_p5
df_trace_win_with_thresh['is_duration_anom'] = (
    df_trace_win_with_thresh['trace_name'].str.contains('duration', na=False) &
    (df_trace_win_with_thresh['value'] > df_trace_win_with_thresh['global_p95'])
)
df_trace_win_with_thresh['is_rowcount_anom'] = (
    df_trace_win_with_thresh['trace_name'].str.contains('row_count', na=False) &
    (df_trace_win_with_thresh['value'] <= df_trace_win_with_thresh['global_p5'])
)

# Compute group-wise window stats
grp = df_trace_win_with_thresh.groupby(['cmdb_id', 'trace_name'], as_index=False)
trace_window_stats = grp.agg(
    window_count=('value', 'count'),
    anomaly_count_duration=('is_duration_anom', 'sum'),
    anomaly_count_rowcount=('is_rowcount_anom', 'sum'),
    max_in_window=('value', 'max'),
    mean_in_window=('value', 'mean')
)

# earliest_anom_ts: earliest timestamp among any anomaly types in window (duration or rowcount)
anom_points = df_trace_win_with_thresh[
    df_trace_win_with_thresh['is_duration_anom'] | df_trace_win_with_thresh['is_rowcount_anom']
].copy()

earliest_anom = (
    anom_points.groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(earliest_anom_ts=('timestamp', 'min'))
)

# Merge earliest_anom into window stats
trace_window_stats = pd.merge(trace_window_stats, earliest_anom, on=['cmdb_id', 'trace_name'], how='left')

# 3) Merge global thresholds with window stats and prepare final report
df_trace_report = pd.merge(
    trace_agg,
    trace_window_stats,
    on=['cmdb_id', 'trace_name'],
    how='left'
)

# Fill missing window-related fields for groups with no points in window
df_trace_report['window_count'] = df_trace_report['window_count'].fillna(0).astype(int)
df_trace_report['anomaly_count_duration'] = df_trace_report['anomaly_count_duration'].fillna(0).astype(int)
df_trace_report['anomaly_count_rowcount'] = df_trace_report['anomaly_count_rowcount'].fillna(0).astype(int)
# max/mean may remain NaN when no window points; keep as-is

# Convert earliest_anom_ts to localized string or None
def epoch_to_local_str(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

df_trace_report['earliest_anom_ts'] = df_trace_report['earliest_anom_ts'].apply(epoch_to_local_str)

# total anomaly count for sorting
df_trace_report['total_anomalies'] = df_trace_report['anomaly_count_duration'] + df_trace_report['anomaly_count_rowcount']

# Select and order columns as requested
report_cols = [
    'cmdb_id', 'trace_name', 'series_count', 'global_p95', 'global_p5',
    'window_count', 'anomaly_count_duration', 'anomaly_count_rowcount',
    'earliest_anom_ts', 'max_in_window', 'mean_in_window', 'total_anomalies'
]
df_trace_report = df_trace_report[report_cols]

# Sort by total anomaly count descending and limit to top 20
df_trace_report_sorted = df_trace_report.sort_values(['total_anomalies', 'window_count', 'series_count'], ascending=[False, False, False])
df_trace_report_top20 = df_trace_report_sorted.head(20).reset_index(drop=True)

# 4) Find overlap between metric-anomalous-components and trace-anomaly results
# Metric-anomalous-components: reuse df_anom_summary (from previous metric run) if available; otherwise derive from df_report
try:
    metric_anom_components = set(df_anom_summary['cmdb_id'].unique())
except NameError:
    # derive from df_report (metric report) where anomaly_count_p95 > 0
    metric_anom_components = set(df_report[df_report['anomaly_count_p95'] > 0]['cmdb_id'].unique())

# Trace-anomalous components: components that have any trace total_anomalies > 0
trace_anom_df = df_trace_report[df_trace_report['total_anomalies'] > 0].copy()

df_trace_anom_summary = (
    trace_anom_df.groupby('cmdb_id', as_index=False)
    .agg(anomalous_trace_count=('trace_name', 'nunique'),
         total_trace_anomaly_points=('total_anomalies', 'sum'))
)

# Overlap components
df_overlap = df_trace_anom_summary[df_trace_anom_summary['cmdb_id'].isin(metric_anom_components)].copy()
df_overlap = df_overlap.sort_values(['anomalous_trace_count', 'total_trace_anomaly_points'], ascending=[False, False]).head(20).reset_index(drop=True)

# Final displayed variables (compact): top 20 trace report rows, and overlap summary
df_trace_report_top20, df_overlap

```
Out[3]:
```
import pandas as pd

# Build a concise plain-English summary using existing kernel variables
for var in ['df_trace_report', 'df_trace_report_top20', 'df_overlap']:
    if var not in globals():
        raise NameError(f"Required variable '{var}' not found in kernel state.")

# Aggregate totals
trace_anom_df = df_trace_report[df_trace_report['total_anomalies'] > 0].copy()
num_trace_anom_groups = int(trace_anom_df.shape[0])
total_trace_anomaly_points = int(trace_anom_df['total_anomalies'].sum())

# Examples (up to 5) from top 20 trace report
examples = []
for _, row in df_trace_report_top20.head(5).iterrows():
    examples.append(
        f"{row['cmdb_id']} | {row['trace_name']} — duration_anoms:{int(row['anomaly_count_duration'])}, "
        f"rowcount_anoms:{int(row['anomaly_count_rowcount'])}, earliest:{row['earliest_anom_ts']}, "
        f"max:{row['max_in_window']:.6g}, mean:{row['mean_in_window']:.6g}"
    )

# Overlap components (from df_overlap) — up to top 20 (already limited)
overlap_list = []
for _, r in df_overlap.iterrows():
    overlap_list.append(f"{r['cmdb_id']}: {int(r['anomalous_trace_count'])} distinct anomalous traces, {int(r['total_trace_anomaly_points'])} anomaly points")

summary = (
    "Trace anomaly analysis summary (thresholds computed on full trace series):\n\n"
    f"- Number of trace component-trace_name groups with anomalies in the incident window: {num_trace_anom_groups}\n"
    f"- Total trace anomaly points observed in the window: {total_trace_anomaly_points}\n\n"
    "- Example anomalous trace entries (up to 5):\n  " + "\n  ".join(examples) + "\n\n"
    "- Components appearing in both metric-anomalous-components and trace-anomaly results (top entries):\n  " + ("\n  ".join(overlap_list) if overlap_list else "(none)") + "\n\n"
    "Notes: 'duration' anomalies counted when value > global_p95; 'row_count' anomalies counted when value <= global_p5. "
    "Global thresholds (p95/p5) were computed on the full trace series before filtering to the incident window."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                       trace_name  series_count   global_p95    global_p5  window_count  anomaly_count_duration  anomaly_count_rowcount         earliest_anom_ts  max_in_window  mean_in_window  total_anomalies
0       MG01    trace.from_dockerA1.row_count            31    74.000000    19.000000            31                       0                       3  2021-03-25 09:06:00 CST      95.000000       42.225806                3
1   dockerA1          trace.to_MG01.row_count            31    74.000000    19.000000            31                       0                       3  2021-03-25 09:06:00 CST      95.000000       42.225806                3
2   dockerA2             trace.self.row_count            31  3275.000000  1494.000000            31                       0                       3  2021-03-25 09:00:00 CST    3579.000000     2312.290323                3
3       IG01         trace.self.duration_mean            31     0.738230     0.259734            31                       2                       0  2021-03-25 09:13:00 CST       0.809721        0.423564                2
4       IG01          trace.self.duration_p95            31     3.579925     1.075250            31                       2                       0  2021-03-25 09:13:00 CST       4.116500        2.014589                2
5       IG01             trace.self.row_count            31  2248.000000  1518.500000            31                       0                       2  2021-03-25 09:05:00 CST    2334.000000     1789.000000                2
6       IG01  trace.to_Tomcat01.duration_mean            31     0.740778     0.234107            31                       2                       0  2021-03-25 09:13:00 CST       0.908046        0.416579                2
7       IG01   trace.to_Tomcat01.duration_p95            31     3.430550     1.046450            31                       2                       0  2021-03-25 09:13:00 CST       5.472400        1.968634                2
8       IG01      trace.to_Tomcat01.row_count            31   293.000000   190.000000            31                       0                       2  2021-03-25 09:04:00 CST     312.000000      226.677419                2
9       IG01  trace.to_Tomcat02.duration_mean            31     0.706363     0.257726            31                       2                       0  2021-03-25 09:13:00 CST       0.857555        0.420136                2
10      IG01   trace.to_Tomcat02.duration_p95            31     3.120625     1.042125            31                       2                       0  2021-03-25 09:13:00 CST       4.119600        1.848179                2
11      IG01      trace.to_Tomcat02.row_count            31   285.000000   188.000000            31                       0                       2  2021-03-25 09:13:00 CST     298.000000      224.483871                2
12      IG01  trace.to_Tomcat03.duration_mean            31     0.704919     0.254866            31                       2                       0  2021-03-25 09:13:00 CST       0.789316        0.412200                2
13      IG01   trace.to_Tomcat03.duration_p95            31     3.141850     1.048950            31                       2                       0  2021-03-25 09:13:00 CST       4.080650        1.924798                2
14      IG01      trace.to_Tomcat03.row_count            31   280.000000   173.000000            31                       0                       2  2021-03-25 09:05:00 CST     303.000000      222.000000                2
15      IG01  trace.to_Tomcat04.duration_mean            31     0.736786     0.258394            31                       2                       0  2021-03-25 09:13:00 CST       0.790961        0.433553                2
16      IG01   trace.to_Tomcat04.duration_p95            31     3.166650     1.047700            31                       2                       0  2021-03-25 09:13:00 CST       4.094850        1.994006                2
17      IG01      trace.to_Tomcat04.row_count            31   269.500000   186.500000            31                       0                       2  2021-03-25 09:05:00 CST     273.000000      221.322581                2
18      IG02         trace.self.duration_mean            31     0.733939     0.245054            31                       2                       0  2021-03-25 09:13:00 CST       0.866535        0.414751                2
19      IG02          trace.self.duration_p95            31     3.598000     1.050000            31                       2                       0  2021-03-25 09:13:00 CST       4.708250        1.952960                2,     cmdb_id  anomalous_trace_count  total_trace_anomaly_points
0      MG01                     42                          82
1      MG02                     42                          81
2      IG01                     18                          33
3      IG02                     18                          33
4  Tomcat04                     18                          33
5  dockerA2                     17                          32
6  Tomcat01                     17                          31
7  Tomcat02                     17                          31
8  Tomcat03                     17                          31)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse timezone if present
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Reuse df_log if already loaded, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure correct dtypes
df_log['timestamp'] = pd.to_numeric(df_log['timestamp'], errors='coerce').astype('Int64')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# 1) Global thresholds on full log series grouped by cmdb_id and log_name
log_agg = (
    df_log
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        series_count=('value', 'count'),
        global_p95=('value', lambda x: x.quantile(0.95) if x.notna().any() else float('nan')),
        global_p5 =('value', lambda x: x.quantile(0.05) if x.notna().any() else float('nan')),
        series_min=('value', 'min'),
        series_max=('value', 'max')
    )
)
log_agg['series_count'] = log_agg['series_count'].astype(int)

# 2) Filter to incident window and compute window metrics and anomalies
start_ts = 1616634000
end_ts = 1616635800
df_log_window = df_log[(df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)].copy()

# Merge global thresholds into windowed points
df_log_win_with_thresh = pd.merge(
    df_log_window,
    log_agg[['cmdb_id', 'log_name', 'global_p95', 'global_p5']],
    on=['cmdb_id', 'log_name'],
    how='left'
)

# Define anomaly conditions:
# - For log_name NOT containing 'row_count': high anomaly when value > global_p95
# - For log_name containing 'row_count': low anomaly when value <= global_p5
df_log_win_with_thresh['is_high_anom'] = (
    ~df_log_win_with_thresh['log_name'].str.contains('row_count', na=False)
) & (df_log_win_with_thresh['value'] > df_log_win_with_thresh['global_p95'])

df_log_win_with_thresh['is_low_anom'] = (
    df_log_win_with_thresh['log_name'].str.contains('row_count', na=False)
) & (df_log_win_with_thresh['value'] <= df_log_win_with_thresh['global_p5'])

# Group windowed results per cmdb_id + log_name
grp = df_log_win_with_thresh.groupby(['cmdb_id', 'log_name'], as_index=False)
df_log_window_stats = grp.agg(
    window_count = ('value', 'count'),
    anomaly_count_high = ('is_high_anom', 'sum'),
    anomaly_count_low = ('is_low_anom', 'sum'),
    max_in_window = ('value', 'max'),
    mean_in_window = ('value', 'mean')
)

# Compute earliest anomaly timestamp per group (either type)
anom_points = df_log_win_with_thresh[df_log_win_with_thresh['is_high_anom'] | df_log_win_with_thresh['is_low_anom']].copy()
earliest_anom = (
    anom_points.groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(earliest_anom_ts=('timestamp', 'min'))
)

df_log_window_stats = pd.merge(df_log_window_stats, earliest_anom, on=['cmdb_id', 'log_name'], how='left')

# 3) Merge global stats with window stats to form final report
df_log_report = pd.merge(
    log_agg,
    df_log_window_stats,
    on=['cmdb_id', 'log_name'],
    how='left'
)

# Fill NaNs for groups that had no points in window
df_log_report['window_count'] = df_log_report['window_count'].fillna(0).astype(int)
df_log_report['anomaly_count_high'] = df_log_report['anomaly_count_high'].fillna(0).astype(int)
df_log_report['anomaly_count_low'] = df_log_report['anomaly_count_low'].fillna(0).astype(int)
# max_in_window and mean_in_window may remain NaN for no-window groups

# Convert earliest_anom_ts epoch -> localized string; keep nulls as None
def epoch_to_local_str(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

df_log_report['earliest_anom_ts'] = df_log_report['earliest_anom_ts'].apply(epoch_to_local_str)

# total anomalies for sorting
df_log_report['total_anomalies'] = df_log_report['anomaly_count_high'] + df_log_report['anomaly_count_low']

# Select requested columns and sort
report_cols = [
    'cmdb_id', 'log_name', 'series_count', 'global_p95', 'global_p5',
    'window_count', 'anomaly_count_high', 'anomaly_count_low',
    'earliest_anom_ts', 'max_in_window', 'mean_in_window', 'total_anomalies'
]
df_log_report = df_log_report[report_cols]

df_log_report_sorted = df_log_report.sort_values(['total_anomalies', 'window_count', 'series_count'], ascending=[False, False, False])
df_log_report_top20 = df_log_report_sorted.head(20).reset_index(drop=True)

# 4) Overlap with metric-anomalous-components from previous metric run
# Reuse df_anom_summary if available, otherwise derive from df_report
try:
    metric_anom_components = set(df_anom_summary['cmdb_id'].unique())
except NameError:
    metric_anom_components = set(df_report[df_report['anomaly_count_p95'] > 0]['cmdb_id'].unique())

# Summarize log anomalies by cmdb_id
df_log_anom_summary = (
    df_log_report[df_log_report['total_anomalies'] > 0]
    .groupby('cmdb_id', as_index=False)
    .agg(distinct_anomalous_log_names=('log_name', 'nunique'),
         total_log_anomaly_points=('total_anomalies', 'sum'))
    .sort_values(['distinct_anomalous_log_names', 'total_log_anomaly_points'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)

# Overlap between metric-anomalous-components and log-anomalous components
df_log_overlap = df_log_anom_summary[df_log_anom_summary['cmdb_id'].isin(metric_anom_components)].reset_index(drop=True)

# Display compact outputs (DataFrames, top results only)
df_log_report_top20, df_log_overlap

```
Out[4]:
```
summary = (
    "Log analysis summary (global thresholds computed on the full log.csv; window = 1616634000..1616635800 UTC+8):\n\n"
    "- Top anomalous log groups (from the compact report): Most anomalies are in log.row_count for Tomcat01/02/03/04 and apache01/02 "
    "(each showing 2 low anomalies in the incident window). Additionally, log.gc_pause_max shows high anomalies on Tomcat04 (2), "
    "Tomcat02 (1) and Tomcat03 (1). log.error_count shows no anomalies in the window.\n\n"
    "- Counts: 6 components have log anomalies (rows in the compact anomaly summary). Total log anomaly points among those components = 16.\n\n"
    "- Overlap with metric-anomalous-components: the following components appear in both metric and log anomaly lists (top entries):\n"
    "  • Tomcat04 — 2 distinct anomalous log_name(s), 4 log anomaly points\n"
    "  • Tomcat02 — 2 distinct anomalous log_name(s), 3 log anomaly points\n"
    "  • Tomcat03 — 2 distinct anomalous log_name(s), 3 log anomaly points\n"
    "  • Tomcat01 — 1 distinct anomalous log_name, 2 log anomaly points\n"
    "  • apache01 — 1 distinct anomalous log_name, 2 log anomaly points\n"
    "  • apache02 — 1 distinct anomalous log_name, 2 log anomaly points\n\n"
    "Notes: 'high' anomalies were counted for non-'row_count' logs when value > global_p95; 'low' anomalies were counted for 'row_count' logs when value <= global_p5. "
    "Global p95/p5 were computed on the full series prior to window filtering."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id          log_name  series_count  global_p95  global_p5  window_count  anomaly_count_high  anomaly_count_low         earliest_anom_ts  max_in_window  mean_in_window  total_anomalies
0   Tomcat01     log.row_count            31     578.500    384.500            31                   0                  2  2021-03-25 09:13:00 CST         598.00      455.225806                2
1   Tomcat02     log.row_count            31     569.500    378.000            31                   0                  2  2021-03-25 09:05:00 CST         597.00      450.096774                2
2   Tomcat03     log.row_count            31     561.000    384.000            31                   0                  2  2021-03-25 09:05:00 CST         619.00      452.258065                2
3   Tomcat04     log.row_count            31     526.000    378.500            31                   0                  2  2021-03-25 09:05:00 CST         537.00      445.580645                2
4   apache01     log.row_count            31    1118.500    783.000            31                   0                  2  2021-03-25 09:05:00 CST        1143.00      896.774194                2
5   apache02     log.row_count            31    1131.500    769.000            31                   0                  2  2021-03-25 09:05:00 CST        1150.00      903.419355                2
6   Tomcat04  log.gc_pause_max            23       0.089      0.020            23                   2                  0  2021-03-25 09:02:00 CST           0.10        0.062174                2
7   Tomcat02  log.gc_pause_max            23       0.100      0.020            23                   1                  0  2021-03-25 09:25:00 CST           0.17        0.068261                1
8   Tomcat03  log.gc_pause_max            22       0.080      0.020            22                   1                  0  2021-03-25 09:28:00 CST           0.09        0.057273                1
9   Tomcat01   log.error_count            31       0.000      0.000            31                   0                  0                     None           0.00        0.000000                0
10  Tomcat02   log.error_count            31       0.000      0.000            31                   0                  0                     None           0.00        0.000000                0
11  Tomcat03   log.error_count            31       0.000      0.000            31                   0                  0                     None           0.00        0.000000                0
12  Tomcat04   log.error_count            31       0.000      0.000            31                   0                  0                     None           0.00        0.000000                0
13  apache01   log.error_count            31       0.000      0.000            31                   0                  0                     None           0.00        0.000000                0
14  apache02   log.error_count            31       0.000      0.000            31                   0                  0                     None           0.00        0.000000                0
15  Tomcat01  log.gc_pause_max            23       0.080      0.021            23                   0                  0                     None           0.08        0.056087                0,     cmdb_id  distinct_anomalous_log_names  total_log_anomaly_points
0  Tomcat04                             2                         4
1  Tomcat02                             2                         3
2  Tomcat03                             2                         3
3  Tomcat01                             1                         2
4  apache01                             1                         2
5  apache02                             1                         2)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse timezone and previous aggregates if present
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure required variables exist in kernel
for var in ['df_series_stats', 'df_window_stats']:
    if var not in globals():
        raise NameError(f"Required variable '{var}' not found. Please run the metric aggregation step first.")

# Merge full-series stats with window stats to get all needed fields (do not recompute thresholds)
df_metric_full = pd.merge(
    df_series_stats[['cmdb_id', 'kpi_name', 'series_count', 'global_p95']],
    df_window_stats[['cmdb_id', 'kpi_name', 'window_count', 'anomaly_count_p95', 'earliest_anom_ts_p95', 'max_in_window', 'mean_in_window']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Focus on the four Tomcats and only anomalous KPI groups (anomaly_count_p95 > 0)
tomcats = ['Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04']
df_tomcats_anom = df_metric_full[
    (df_metric_full['cmdb_id'].isin(tomcats)) &
    (df_metric_full['anomaly_count_p95'].fillna(0) > 0)
].copy()

# Ensure numeric types and fill window-related NaNs reasonably (window_count should be int)
df_tomcats_anom['window_count'] = df_tomcats_anom['window_count'].fillna(0).astype(int)
df_tomcats_anom['anomaly_count_p95'] = df_tomcats_anom['anomaly_count_p95'].fillna(0).astype(int)

# Keep epoch column for sorting; earliest_anom_ts_p95 in df_window_stats is epoch (or NaN)
df_tomcats_anom['earliest_anom_epoch'] = pd.to_numeric(df_tomcats_anom['earliest_anom_ts_p95'], errors='coerce')

# Convert earliest anomaly epoch to localized string for display
def epoch_to_local_str(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

df_tomcats_anom['earliest_anom_ts_p95_local'] = df_tomcats_anom['earliest_anom_epoch'].apply(epoch_to_local_str)

# Select requested columns and sort by earliest anomaly ascending (nulls last)
display_cols = [
    'cmdb_id', 'kpi_name', 'series_count', 'global_p95',
    'window_count', 'anomaly_count_p95', 'earliest_anom_ts_p95_local',
    'max_in_window', 'mean_in_window'
]
df_tomcats_anom_display = df_tomcats_anom[display_cols].copy()
df_tomcats_anom_display = df_tomcats_anom_display.assign(
    _sort_epoch = df_tomcats_anom['earliest_anom_epoch']
).sort_values('_sort_epoch', na_position='last').drop(columns=['_sort_epoch']).reset_index(drop=True)

# 2nd aggregated table: per Tomcat summary (distinct anomalous KPI count, total anomaly points, earliest anomaly across KPIs)
df_tomcats_agg = (
    df_tomcats_anom
    .groupby('cmdb_id', as_index=False)
    .agg(
        distinct_anomalous_kpi_count=('kpi_name', 'nunique'),
        total_anomaly_points=('anomaly_count_p95', 'sum'),
        earliest_anom_epoch=('earliest_anom_epoch', 'min')
    )
)

df_tomcats_agg['earliest_anom_ts_local'] = df_tomcats_agg['earliest_anom_epoch'].apply(epoch_to_local_str)

# Keep compact columns and sort by earliest anomaly ascending
df_tomcats_agg_display = df_tomcats_agg[['cmdb_id', 'distinct_anomalous_kpi_count', 'total_anomaly_points', 'earliest_anom_ts_local']].sort_values('earliest_anom_ts_local', na_position='last').reset_index(drop=True)

# Limit rows to keep output compact (request allows up to 100; these sets are small)
df_tomcats_anom_display, df_tomcats_agg_display

```
Out[5]:
```
summary = (
    "Tomcat metric anomaly summary (incident window 2021-03-25 09:00:00–09:30:00 CST):\n\n"
    "- Total anomalous KPI groups across Tomcat01/02/03/04: 177 rows.\n"
    "- Total anomaly points across these Tomcats: 213 points (sum of per-KPI anomaly_count_p95).\n\n"
    "Per-Tomcat summary:\n"
    "  • Tomcat01 — 50 distinct anomalous KPIs, 59 anomaly points, earliest anomaly at 2021-03-25 09:00:00 CST\n"
    "  • Tomcat02 — 42 distinct anomalous KPIs, 51 anomaly points, earliest anomaly at 2021-03-25 09:00:00 CST\n"
    "  • Tomcat03 — 40 distinct anomalous KPIs, 49 anomaly points, earliest anomaly at 2021-03-25 09:00:00 CST\n"
    "  • Tomcat04 — 45 distinct anomalous KPIs, 54 anomaly points, earliest anomaly at 2021-03-25 09:02:00 CST\n\n"
    "Timing insight: earliest anomalies for Tomcat01/02/03 occur at ~09:00 and for Tomcat04 at ~09:02. "
    "Anomalies span the entire incident window (09:00–09:30); there are KPI-level anomalies near 09:13 as well, "
    "but the per-Tomcat earliest times above are earlier than 09:13.\n\n"
    "If you want the detailed per-KPI rows (sorted by earliest anomaly time) they are available in kernel variable "
    "'df_tomcats_anom_display' (177 rows). A compact per-Tomcat aggregation is in 'df_tomcats_agg_display'."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                          kpi_name  series_count    global_p95  window_count  anomaly_count_p95 earliest_anom_ts_p95_local  max_in_window  mean_in_window
0    Tomcat01                       OSLinux-CPU_CPU_CPUidleutil            31     72.997773            31                  2    2021-03-25 09:00:00 CST      73.236900       68.718147
1    Tomcat02                            OSLinux-CPU_CPU_CPUWio            30      0.377781            30                  2    2021-03-25 09:00:00 CST       0.528330        0.115395
2    Tomcat03                            OSLinux-CPU_CPU_CPUWio            31      0.532474            31                  2    2021-03-25 09:00:00 CST       1.198200        0.228044
3    Tomcat02                   OSLinux-CPU_CPU-0_SingleCpuUtil            10     92.007820            10                  1    2021-03-25 09:01:00 CST     100.000000       21.207220
4    Tomcat02                   OSLinux-CPU_CPU-3_SingleCpuidle            15     24.048800            15                  1    2021-03-25 09:01:00 CST      79.813600        5.330880
..        ...                                               ...           ...           ...           ...                ...                        ...            ...             ...
172  Tomcat02  OSLinux-OSLinux_FILESYSTEM_----tomcat_FSCapacity            15     75.343755            15                  1    2021-03-25 09:30:00 CST      75.363421       75.188722
173  Tomcat04  OSLinux-OSLinux_FILESYSTEM_----tomcat_FSCapacity            15     50.404077            15                  1    2021-03-25 09:30:00 CST      50.422675       50.252831
174  Tomcat04    OSLinux-OSLinux_FILESYSTEM_-tomcat_FSUsedSpace            11  23976.500000            11                  1    2021-03-25 09:30:00 CST   23983.000000    23909.000000
175  Tomcat04     OSLinux-OSLinux_FILESYSTEM_----tmp_FSCapacity            15      0.829754            15                  1    2021-03-25 09:30:00 CST       0.829812        0.829735
176  Tomcat04  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite            15    273.773310            15                  1    2021-03-25 09:30:00 CST     292.533333      212.711107

[177 rows x 9 columns],     cmdb_id  distinct_anomalous_kpi_count  total_anomaly_points   earliest_anom_ts_local
0  Tomcat01                            50                    59  2021-03-25 09:00:00 CST
1  Tomcat02                            42                    51  2021-03-25 09:00:00 CST
2  Tomcat03                            40                    49  2021-03-25 09:00:00 CST
3  Tomcat04                            45                    54  2021-03-25 09:02:00 CST)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Try to reuse existing tomcats anomalous table; if not available, recompute from metric aggregates
if 'df_tomcats_anom_display' in globals():
    df_all_tomcat_anom = df_tomcats_anom_display.copy()
    # try to recover epoch for robust sorting if available in df_tomcats_anom
    if 'df_tomcats_anom' in globals() and 'earliest_anom_epoch' in df_tomcats_anom.columns:
        epoch_lookup = df_tomcats_anom[['cmdb_id', 'kpi_name', 'earliest_anom_epoch']]
        df_all_tomcat_anom = pd.merge(df_all_tomcat_anom, epoch_lookup, on=['cmdb_id', 'kpi_name'], how='left')
    else:
        # try to parse the local timestamp string into epoch (may be None)
        df_all_tomcat_anom['earliest_anom_epoch'] = pd.to_datetime(
            df_all_tomcat_anom['earliest_anom_ts_p95_local'], utc=True, errors='coerce'
        ).astype('Int64') // 10**9
else:
    # Recompute minimal required tables from metric full-series and window stats (reuse df_series_stats and df_window_stats if present)
    if 'df_series_stats' not in globals() or 'df_window_stats' not in globals():
        # load metric.csv and recompute series and window aggregates as needed
        df_metric = pd.read_csv('metric.csv')
        df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
        df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
        # full series stats
        df_series_stats = (
            df_metric.groupby(['cmdb_id','kpi_name'], as_index=False)
            .agg(series_count=('value','count'),
                 global_p95=('value', lambda x: x.quantile(0.95)),
                 global_p90=('value', lambda x: x.quantile(0.90)),
                 global_p5=('value', lambda x: x.quantile(0.05)),
                 series_min=('value','min'),
                 series_max=('value','max'))
        )
        df_series_stats['series_count'] = df_series_stats['series_count'].astype(int)
        # windowed stats
        start_ts = 1616634000
        end_ts = 1616635800
        df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
        df_window = pd.merge(df_window, df_series_stats[['cmdb_id','kpi_name','global_p95']], on=['cmdb_id','kpi_name'], how='left')
        df_window['is_anom_p95'] = df_window['value'] > df_window['global_p95']
        grp = df_window.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
            window_count=('value','count'),
            anomaly_count_p95=('is_anom_p95','sum'),
            max_in_window=('value','max'),
            mean_in_window=('value','mean')
        )
        earliest = df_window[df_window['is_anom_p95']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(earliest_anom_ts_p95=('timestamp','min'))
        df_window_stats = pd.merge(grp, earliest, on=['cmdb_id','kpi_name'], how='left')
    # merge to create the display DF
    df_full = pd.merge(
        df_series_stats[['cmdb_id','kpi_name','series_count','global_p95']],
        df_window_stats[['cmdb_id','kpi_name','window_count','anomaly_count_p95','earliest_anom_ts_p95','max_in_window','mean_in_window']],
        on=['cmdb_id','kpi_name'], how='left'
    )
    df_full['window_count'] = df_full['window_count'].fillna(0).astype(int)
    df_full['anomaly_count_p95'] = df_full['anomaly_count_p95'].fillna(0).astype(int)
    # convert earliest epoch to localized string
    def epoch_to_local_str(x):
        if pd.isna(x):
            return None
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    df_full['earliest_anom_ts_p95_local'] = df_full['earliest_anom_ts_p95'].apply(epoch_to_local_str)
    df_full['earliest_anom_epoch'] = pd.to_numeric(df_full['earliest_anom_ts_p95'], errors='coerce')
    df_all_tomcat_anom = df_full[['cmdb_id','kpi_name','series_count','global_p95','window_count','anomaly_count_p95','earliest_anom_ts_p95_local','max_in_window','mean_in_window','earliest_anom_epoch']].copy()

# Now filter for Tomcat01 and only anomalous KPI groups (anomaly_count_p95 > 0)
df_tomcat01 = df_all_tomcat_anom[
    (df_all_tomcat_anom['cmdb_id'] == 'Tomcat01') &
    (pd.to_numeric(df_all_tomcat_anom['anomaly_count_p95'], errors='coerce').fillna(0) > 0)
].copy()

# Ensure numeric types
df_tomcat01['global_p95'] = pd.to_numeric(df_tomcat01['global_p95'], errors='coerce')
df_tomcat01['max_in_window'] = pd.to_numeric(df_tomcat01['max_in_window'], errors='coerce')

# Compute severity_ratio = max_in_window / global_p95 (null if global_p95 == 0 or NaN)
df_tomcat01['severity_ratio'] = np.where(
    (df_tomcat01['global_p95'].fillna(0) == 0),
    np.nan,
    df_tomcat01['max_in_window'] / df_tomcat01['global_p95']
)

# For sorting, ensure we have epoch column; try to use 'earliest_anom_epoch' if present, else parse local string
if 'earliest_anom_epoch' not in df_tomcat01.columns:
    df_tomcat01['earliest_anom_epoch'] = pd.to_datetime(df_tomcat01['earliest_anom_ts_p95_local'], utc=True, errors='coerce').astype('Int64') // 10**9

# Sort by earliest_anom_epoch ascending (NaNs last) then severity_ratio descending
df_tomcat01_sorted = df_tomcat01.sort_values(['earliest_anom_epoch','severity_ratio'], ascending=[True, False], na_position='last')

# Select requested columns and limit to top 20
display_cols = ['cmdb_id','kpi_name','series_count','global_p95','window_count','anomaly_count_p95','earliest_anom_ts_p95_local','max_in_window','mean_in_window','severity_ratio']
df_tomcat01_top20 = df_tomcat01_sorted[display_cols].head(20).reset_index(drop=True)

# Also return KPI groups whose earliest_anom_ts_p95_local == '2021-03-25 09:13:00 CST'
target_time = '2021-03-25 09:13:00 CST'
df_tomcat01_at_0913 = df_tomcat01_top20[df_tomcat01_top20['earliest_anom_ts_p95_local'] == target_time][['kpi_name','severity_ratio']].reset_index(drop=True)

# Display compact results (DataFrames)
df_tomcat01_top20, df_tomcat01_at_0913

```
Out[6]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure required variables exist
if 'df_tomcat01' not in globals():
    raise NameError("Required variable 'df_tomcat01' not found in kernel state.")
if 'df_tomcat01_top20' not in globals():
    raise NameError("Required variable 'df_tomcat01_top20' not found in kernel state.")
if 'df_tomcat01_at_0913' not in globals():
    # create an empty placeholder if missing
    df_tomcat01_at_0913 = pd.DataFrame(columns=['kpi_name','severity_ratio'])

# Aggregate totals for Tomcat01 anomalous KPIs
total_kpis = int(df_tomcat01.shape[0])
total_anomaly_points = int(pd.to_numeric(df_tomcat01['anomaly_count_p95'], errors='coerce').fillna(0).sum())

# earliest anomaly epoch -> localized string
earliest_epoch = pd.to_numeric(df_tomcat01.get('earliest_anom_epoch', pd.Series([], dtype='float')), errors='coerce').min()
if pd.isna(earliest_epoch):
    earliest_local = None
else:
    earliest_local = pd.to_datetime(int(earliest_epoch), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

# KPIs that first appear exactly at 2021-03-25 09:13:00 CST
target_time = '2021-03-25 09:13:00 CST'
kpis_at_0913 = df_tomcat01_at_0913.copy()
if kpis_at_0913.empty:
    kpis_at_0913_list = "(none)"
else:
    kpis_at_0913_list = "; ".join(kpis_at_0913.apply(lambda r: f"{r['kpi_name']} (severity_ratio={r['severity_ratio']:.3g})", axis=1).tolist())

summary = (
    "Tomcat01 anomaly summary (metric KPIs with anomaly_count_p95 > 0):\n\n"
    f"- Total distinct anomalous KPI groups: {total_kpis}\n"
    f"- Total anomaly points (sum of anomaly_count_p95): {total_anomaly_points}\n"
    f"- Earliest anomaly observed for Tomcat01: {earliest_local}\n\n"
    "- A compact top-20 table of these anomalous KPI groups is available in kernel variable 'df_tomcat01_top20'.\n"
    "  Columns include: cmdb_id, kpi_name, series_count, global_p95, window_count, anomaly_count_p95, "
    "earliest_anom_ts_p95_local, max_in_window, mean_in_window, severity_ratio (max_in_window / global_p95).\n\n"
    f"- KPI groups whose earliest_anom_ts_p95_local == {target_time}: {kpis_at_0913_list}\n\n"
    "Notes: severity_ratio is computed as max_in_window / global_p95 (NaN if global_p95 == 0)."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  series_count    global_p95  window_count  anomaly_count_p95 earliest_anom_ts_p95_local  max_in_window  mean_in_window  severity_ratio
0   Tomcat01                        OSLinux-CPU_CPU_CPUidleutil            31  7.299777e+01            31                  2    2021-03-25 09:00:00 CST   7.323690e+01    6.871815e+01        1.003276
1   Tomcat01  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...            10  1.774270e+04            10                  1    2021-03-25 09:01:00 CST   1.776700e+04    1.751260e+04        1.001370
2   Tomcat01                    OSLinux-CPU_CPU-2_SingleCpuidle            15  8.066800e+01            15                  1    2021-03-25 09:03:00 CST   9.827160e+01    1.159695e+01        1.218223
3   Tomcat01                    OSLinux-CPU_CPU-1_SingleCpuUtil            10  7.213717e+01            10                  1    2021-03-25 09:04:00 CST   9.976740e+01    1.642666e+01        1.383023
4   Tomcat01       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc            20  5.034213e+01            20                  1    2021-03-25 09:04:00 CST   5.037870e+01    4.995955e+01        1.000727
5   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead            15  2.133331e-01            15                  1    2021-03-25 09:05:00 CST   4.000000e-01    4.444220e-02        1.875002
6   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps            15  2.669000e-02            15                  1    2021-03-25 09:05:00 CST   5.000000e-02    5.557800e-03        1.873361
7   Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...            15  2.690000e+00            15                  1    2021-03-25 09:05:00 CST   4.300000e+00    4.866667e-01        1.598513
8   Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps            15  1.120000e+01            15                  1    2021-03-25 09:05:00 CST   1.400000e+01    2.400000e+00        1.250000
9   Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps            15  7.100000e-01            15                  1    2021-03-25 09:05:00 CST   8.500000e-01    1.055555e-01        1.197183
10  Tomcat01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWTps            15  7.951000e+01            15                  1    2021-03-25 09:05:00 CST   8.525000e+01    1.490889e+01        1.072192
11  Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKWrite            15  9.923728e+04            15                  1    2021-03-25 09:05:00 CST   1.053321e+05    1.769820e+04        1.061417
12  Tomcat01   OSLinux-OSLinux_FILESYSTEM_----tomcat_FSCapacity            14  6.441148e+01            14                  1    2021-03-25 09:05:00 CST   6.477257e+01    6.305024e+01        1.005606
13  Tomcat01     OSLinux-OSLinux_FILESYSTEM_-tomcat_FSUsedSpace            10  3.068965e+04            10                  1    2021-03-25 09:05:00 CST   3.080800e+04    3.005090e+04        1.003856
14  Tomcat01             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem            19  3.737200e+03            19                  1    2021-03-25 09:05:00 CST   3.748000e+03    3.087158e+03        1.002890
15  Tomcat01              OSLinux-OSLinux_MEMORY_MEMORY_UserMem            19  7.660800e+03            19                  1    2021-03-25 09:05:00 CST   7.668000e+03    6.979053e+03        1.000940
16  Tomcat01                             OSLinux-CPU_CPU_CPUWio            31  2.307420e+01            31                  2    2021-03-25 09:05:00 CST   2.308396e+01    3.838121e+00        1.000423
17  Tomcat01                         OSLinux-CPU_CPU_CPUSysTime            31  2.074471e+00            31                  2    2021-03-25 09:06:00 CST   2.237099e+00    9.743519e-01        1.078395
18  Tomcat01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc            22  9.600000e+01            22                  1    2021-03-25 09:06:00 CST   9.800000e+01    8.886364e+01        1.020833
19  Tomcat01            Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory             6  1.863259e+09             6                  1    2021-03-25 09:06:00 CST   1.880032e+09    1.662852e+09        1.009002, Empty DataFrame
Columns: [kpi_name, severity_ratio]
Index: [])```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing aggregated variables if available; otherwise recompute from metric.csv
if 'df_series_stats' not in globals() or 'df_window_stats' not in globals():
    # Load metric.csv and compute full-series thresholds and windowed anomalies (as done previously)
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

    # Full-series stats
    df_series_stats = (
        df_metric.groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(
            series_count=('value','count'),
            global_p95=('value', lambda x: x.quantile(0.95) if x.notna().any() else np.nan),
            global_p90=('value', lambda x: x.quantile(0.90) if x.notna().any() else np.nan),
            global_p5 =('value', lambda x: x.quantile(0.05) if x.notna().any() else np.nan),
            series_min=('value','min'),
            series_max=('value','max')
        )
    )
    df_series_stats['series_count'] = df_series_stats['series_count'].astype(int)

    # Windowed (incident) stats
    start_ts = 1616634000
    end_ts = 1616635800
    df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
    df_window = pd.merge(df_window, df_series_stats[['cmdb_id','kpi_name','global_p95']], on=['cmdb_id','kpi_name'], how='left')
    df_window['is_anom_p95'] = df_window['value'] > df_window['global_p95']

    df_window_stats = df_window.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        window_count=('value','count'),
        anomaly_count_p95=('is_anom_p95','sum'),
        max_in_window=('value','max'),
        mean_in_window=('value','mean')
    )
    earliest = df_window[df_window['is_anom_p95']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(earliest_anom_ts_p95=('timestamp','min'))
    df_window_stats = pd.merge(df_window_stats, earliest, on=['cmdb_id','kpi_name'], how='left')

# Merge full-series and window stats for IG01
df_ig01 = pd.merge(
    df_series_stats[['cmdb_id','kpi_name','series_count','global_p95']],
    df_window_stats[['cmdb_id','kpi_name','window_count','anomaly_count_p95','earliest_anom_ts_p95','max_in_window','mean_in_window']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Filter to IG01 and anomaly_count_p95 > 0
df_ig01 = df_ig01[df_ig01['cmdb_id'] == 'IG01'].copy()
df_ig01['anomaly_count_p95'] = pd.to_numeric(df_ig01['anomaly_count_p95'], errors='coerce').fillna(0).astype(int)
df_ig01 = df_ig01[df_ig01['anomaly_count_p95'] > 0].copy()

# Convert earliest epoch to localized string and keep epoch for sorting
def epoch_to_local_str(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

df_ig01['earliest_anom_epoch'] = pd.to_numeric(df_ig01['earliest_anom_ts_p95'], errors='coerce')
df_ig01['earliest_anom_ts_p95_local'] = df_ig01['earliest_anom_epoch'].apply(epoch_to_local_str)

# Ensure numeric fields
df_ig01['global_p95'] = pd.to_numeric(df_ig01['global_p95'], errors='coerce')
df_ig01['max_in_window'] = pd.to_numeric(df_ig01['max_in_window'], errors='coerce')

# Compute severity_ratio = max_in_window / global_p95 (null if global_p95 == 0 or NaN)
df_ig01['severity_ratio'] = np.where(
    (df_ig01['global_p95'].fillna(0) == 0),
    np.nan,
    df_ig01['max_in_window'] / df_ig01['global_p95']
)

# Sort by earliest_anom_ts_p95_local ascending (using epoch) then severity_ratio descending
df_ig01_sorted = df_ig01.sort_values(['earliest_anom_epoch','severity_ratio'], ascending=[True, False], na_position='last')

# Prepare compact top-20 result with requested columns
display_cols = [
    'cmdb_id','kpi_name','series_count','global_p95',
    'window_count','anomaly_count_p95','earliest_anom_ts_p95_local',
    'max_in_window','mean_in_window','severity_ratio'
]
df_ig01_top20 = df_ig01_sorted[display_cols].head(20).reset_index(drop=True)

# KPI groups whose earliest_anom_ts_p95_local == '2021-03-25 09:13:00 CST'
target_time = '2021-03-25 09:13:00 CST'
df_ig01_at_0913 = df_ig01_top20[df_ig01_top20['earliest_anom_ts_p95_local'] == target_time][['kpi_name','severity_ratio']].reset_index(drop=True)

# IG01 aggregate row: distinct_anomalous_kpi_count, total_anomaly_points, earliest_anomaly_ts_local across IG01 KPIs
if not df_ig01.empty:
    distinct_count = int(df_ig01['kpi_name'].nunique())
    total_points = int(df_ig01['anomaly_count_p95'].sum())
    earliest_epoch_all = pd.to_numeric(df_ig01['earliest_anom_epoch'], errors='coerce').min()
    earliest_local_all = epoch_to_local_str(earliest_epoch_all) if not pd.isna(earliest_epoch_all) else None
    df_ig01_agg = pd.DataFrame([{
        'cmdb_id': 'IG01',
        'distinct_anomalous_kpi_count': distinct_count,
        'total_anomaly_points': total_points,
        'earliest_anomaly_ts_local': earliest_local_all
    }])
else:
    df_ig01_agg = pd.DataFrame(columns=['cmdb_id','distinct_anomalous_kpi_count','total_anomaly_points','earliest_anomaly_ts_local'])

# Return compact outputs (DataFrames)
df_ig01_top20, df_ig01_at_0913, df_ig01_agg

```
Out[7]:
```
# Compose a concise plain-English summary using existing kernel variables for IG01
# (expects df_ig01_top20, df_ig01_at_0913, df_ig01_agg to exist in kernel)
import pandas as pd

for var in ['df_ig01_top20', 'df_ig01_at_0913', 'df_ig01_agg']:
    if var not in globals():
        raise NameError(f"Required variable '{var}' not found in kernel state.")

# Basic counts and aggregate info
top_rows_count = df_ig01_top20.shape[0]
agg_row = df_ig01_agg.iloc[0] if not df_ig01_agg.empty else None
distinct_kpis = int(agg_row['distinct_anomalous_kpi_count']) if agg_row is not None else 0
total_anomaly_points = int(agg_row['total_anomaly_points']) if agg_row is not None else 0
earliest_overall = agg_row['earliest_anomaly_ts_local'] if agg_row is not None else None

# Prepare up to 5 example KPI lines from the top-20 table
examples = []
for _, r in df_ig01_top20.head(5).iterrows():
    examples.append(
        f"{r['kpi_name']} — anomalies:{int(r['anomaly_count_p95'])}, earliest:{r['earliest_anom_ts_p95_local']}, severity_ratio:{(r['severity_ratio'] if pd.notna(r['severity_ratio']) else None):.3g}"
    )

# KPIs that first appear exactly at 2021-03-25 09:13:00 CST
if df_ig01_at_0913.empty:
    kpis_at_0913 = "(none)"
else:
    kpis_at_0913 = "; ".join(df_ig01_at_0913.apply(lambda r: f"{r['kpi_name']} (severity_ratio={r['severity_ratio']:.3g})", axis=1).tolist())

summary = (
    "IG01 metric anomaly summary (global p95 thresholds computed on full series; window = 1616634000..1616635800 UTC+8):\n\n"
    f"- Returned top {top_rows_count} anomalous IG01 KPI rows (limited to 20 shown here).\n"
    f"- IG01 aggregates: {distinct_kpis} distinct anomalous KPIs, {total_anomaly_points} total anomaly points; earliest anomaly at {earliest_overall}.\n\n"
    "- Example anomalous KPIs (up to 5):\n  " + ("\n  ".join(examples) if examples else "(none)") + "\n\n"
    f"- KPI groups whose earliest_anom_ts_p95_local == 2021-03-25 09:13:00 CST: {kpis_at_0913}\n\n"
    "Notes: severity_ratio = max_in_window / global_p95 (null if global_p95 == 0). Full top-20 table is available in kernel variable 'df_ig01_top20'; "
    "the aggregate IG01 row is in 'df_ig01_agg'."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           kpi_name  series_count    global_p95  window_count  anomaly_count_p95 earliest_anom_ts_p95_local  max_in_window  mean_in_window  severity_ratio
0     IG01                    OSLinux-CPU_CPU-0_SingleCpuidle             8  9.875209e+01             8                  1    2021-03-25 09:01:00 CST   9.886910e+01    9.822800e+01        1.001185
1     IG01  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...             9  3.741640e+04             9                  1    2021-03-25 09:01:00 CST   3.742400e+04    3.735267e+04        1.000203
2     IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed            31  8.233135e+08            31                  2    2021-03-25 09:05:00 CST   8.565236e+08    4.687445e+08        1.040337
3     IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage            22  4.077394e+01            22                  2    2021-03-25 09:06:00 CST   4.198370e+01    2.013757e+01        1.029670
4     IG01           JVM-Threads_7778_JVM_ThreadCount_Threads            31  7.600000e+01            31                  1    2021-03-25 09:06:00 CST   7.700000e+01    7.212903e+01        1.013158
5     IG01                         OSLinux-CPU_CPU_CPUCpuUtil            19  2.641557e+01            19                  1    2021-03-25 09:09:00 CST   2.662590e+01    2.612479e+01        1.007962
6     IG01                             OSLinux-CPU_CPU_CPUWio            20  6.114600e-01            20                  1    2021-03-25 09:10:00 CST   1.879900e+00    2.597200e-01        3.074445
7     IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...            11  3.055000e+02            11                  1    2021-03-25 09:10:00 CST   5.970000e+02    6.281818e+01        1.954173
8     IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps            10  4.400000e+00            10                  1    2021-03-25 09:10:00 CST   8.000000e+00    8.000000e-01        1.818182
9     IG01                    OSLinux-CPU_CPU-0_SingleCpuUtil            11  1.340055e+01            11                  1    2021-03-25 09:10:00 CST   2.395680e+01    3.660027e+00        1.787748
10    IG01           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem            19  1.545000e+02            19                  1    2021-03-25 09:10:00 CST   1.590000e+02    1.302105e+02        1.029126
11    IG01                        OSLinux-CPU_CPU_CPUidleutil            20  7.411497e+01            20                  1    2021-03-25 09:13:00 CST   7.414860e+01    7.370334e+01        1.000454
12    IG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps             8  9.716450e-01             8                  1    2021-03-25 09:16:00 CST   9.833000e-01    8.520750e-01        1.011995
13    IG01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite             8  1.430666e+01             8                  1    2021-03-25 09:16:00 CST   1.440000e+01    1.173331e+01        1.006525
14    IG01                    OSLinux-CPU_CPU-2_SingleCpuidle             8  9.877101e+01             8                  1    2021-03-25 09:18:00 CST   9.883510e+01    9.839976e+01        1.000649
15    IG01    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec             9  1.397832e+05             9                  1    2021-03-25 09:20:00 CST   1.409020e+05    9.603078e+04        1.008004
16    IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps            10  5.500000e-01            10                  1    2021-03-25 09:21:00 CST   1.000000e+00    1.000000e-01        1.818182
17    IG01    OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum            10  1.419050e+03            10                  1    2021-03-25 09:21:00 CST   1.424000e+03    1.251800e+03        1.003488
18    IG01             OSLinux-OSLinux_MEMORY_MEMORY_CacheMem            19  5.438200e+03            19                  1    2021-03-25 09:22:00 CST   5.440000e+03    5.427158e+03        1.000331
19    IG01              OSLinux-OSLinux_MEMORY_MEMORY_UserMem            19  7.670100e+03            19                  1    2021-03-25 09:22:00 CST   7.671000e+03    7.658421e+03        1.000117,                       kpi_name  severity_ratio
0  OSLinux-CPU_CPU_CPUidleutil        1.000454,   cmdb_id  distinct_anomalous_kpi_count  total_anomaly_points earliest_anomaly_ts_local
0    IG01                            38                    40   2021-03-25 09:01:00 CST)```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse previously computed df_trace_report if available; otherwise recompute from trace.csv
if 'df_trace_report' not in globals():
    # Recompute trace aggregates (full-series thresholds then windowed anomalies) -- minimal reproduction
    df_trace = pd.read_csv('trace.csv')
    df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
    df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
    # full series thresholds
    trace_agg = (
        df_trace.groupby(['cmdb_id','trace_name'], as_index=False)
        .agg(
            series_count=('value','count'),
            global_p95=('value', lambda x: x.quantile(0.95) if x.notna().any() else np.nan),
            global_p5 =('value', lambda x: x.quantile(0.05) if x.notna().any() else np.nan)
        )
    )
    trace_agg['series_count'] = trace_agg['series_count'].astype(int)
    # window filtering and merging
    start_ts = 1616634000
    end_ts = 1616635800
    df_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()
    df_trace_win_with_thresh = pd.merge(df_trace_window, trace_agg[['cmdb_id','trace_name','global_p95','global_p5']], on=['cmdb_id','trace_name'], how='left')
    df_trace_win_with_thresh['is_duration_anom'] = df_trace_win_with_thresh['trace_name'].str.contains('duration', na=False) & (df_trace_win_with_thresh['value'] > df_trace_win_with_thresh['global_p95'])
    df_trace_win_with_thresh['is_rowcount_anom'] = df_trace_win_with_thresh['trace_name'].str.contains('row_count', na=False) & (df_trace_win_with_thresh['value'] <= df_trace_win_with_thresh['global_p5'])
    grp = df_trace_win_with_thresh.groupby(['cmdb_id','trace_name'], as_index=False)
    trace_window_stats = grp.agg(
        window_count=('value','count'),
        anomaly_count_duration=('is_duration_anom','sum'),
        anomaly_count_rowcount=('is_rowcount_anom','sum'),
        max_in_window=('value','max'),
        mean_in_window=('value','mean')
    )
    earliest_anom = df_trace_win_with_thresh[(df_trace_win_with_thresh['is_duration_anom']) | (df_trace_win_with_thresh['is_rowcount_anom'])].groupby(['cmdb_id','trace_name'], as_index=False).agg(earliest_anom_ts=('timestamp','min'))
    trace_window_stats = pd.merge(trace_window_stats, earliest_anom, on=['cmdb_id','trace_name'], how='left')
    # assemble df_trace_report
    df_trace_report = pd.merge(trace_agg, trace_window_stats, on=['cmdb_id','trace_name'], how='left')
    df_trace_report['window_count'] = df_trace_report['window_count'].fillna(0).astype(int)
    df_trace_report['anomaly_count_duration'] = df_trace_report['anomaly_count_duration'].fillna(0).astype(int)
    df_trace_report['anomaly_count_rowcount'] = df_trace_report['anomaly_count_rowcount'].fillna(0).astype(int)
    # localized earliest_anom_ts string
    def epoch_to_local_str(x):
        if pd.isna(x):
            return None
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    df_trace_report['earliest_anom_ts'] = df_trace_report['earliest_anom_ts'].apply(epoch_to_local_str)
    # total anomalies
    df_trace_report['total_anomalies'] = df_trace_report['anomaly_count_duration'] + df_trace_report['anomaly_count_rowcount']

# 1) Compute severity_ratio for trace groups that have anomalies in the incident window
df_trace_anom = df_trace_report[df_trace_report['total_anomalies'] > 0].copy()

# ensure numeric
df_trace_anom['global_p95'] = pd.to_numeric(df_trace_anom['global_p95'], errors='coerce')
df_trace_anom['max_in_window'] = pd.to_numeric(df_trace_anom['max_in_window'], errors='coerce')

# severity_ratio = max_in_window / global_p95 (null if global_p95 == 0 or NaN)
df_trace_anom['severity_ratio'] = np.where(
    df_trace_anom['global_p95'].fillna(0) == 0,
    np.nan,
    df_trace_anom['max_in_window'] / df_trace_anom['global_p95']
)

# Prepare display columns
cols = ['cmdb_id','trace_name','series_count','global_p95','window_count',
        'anomaly_count_duration','anomaly_count_rowcount','earliest_anom_ts',
        'max_in_window','mean_in_window','severity_ratio']

# Sort by severity_ratio descending and limit to top 20
df_severity_top20 = df_trace_anom.sort_values('severity_ratio', ascending=False, na_position='last')[cols].head(20).reset_index(drop=True)

# 2) Focused table: cmdb_id == 'IG01' OR trace_name contains 'Tomcat01'
mask_focus = (df_trace_anom['cmdb_id'] == 'IG01') | (df_trace_anom['trace_name'].str.contains('Tomcat01', na=False))
df_focus = df_trace_anom[mask_focus].copy()

# Sort by earliest_anom_ts ascending (nulls last). earliest_anom_ts is localized string; lexicographic sort works for same format.
df_focus_sorted = df_focus.sort_values('earliest_anom_ts', na_position='last')[cols].head(10).reset_index(drop=True)

# 3) Aggregate: top 5 cmdb_id by mean severity_ratio among their anomalous trace groups
agg = (
    df_trace_anom.groupby('cmdb_id', as_index=False)
    .agg(
        mean_severity_ratio=('severity_ratio', 'mean'),
        total_trace_anomaly_points=('total_anomalies', 'sum'),
        anomalous_trace_groups=('trace_name','nunique')
    )
)

# Only keep cmdb_id with at least one non-NaN severity_ratio (mean will be NaN otherwise) -- but still sort placing NaNs last
agg = agg.dropna(subset=['mean_severity_ratio']).sort_values('mean_severity_ratio', ascending=False).head(5).reset_index(drop=True)

df_top5_by_mean_severity = agg[['cmdb_id','mean_severity_ratio','total_trace_anomaly_points']]

# Compact outputs
df_severity_top20, df_focus_sorted, df_top5_by_mean_severity

```
Out[8]:
```
Summary of trace-anomaly severity analysis (incident window 2021-03-25 09:00–09:30 CST):

1) Top anomalous trace groups by severity_ratio (max_in_window / global_p95)
- Highest severity traces are duration metrics from/to dockerA / MG components, many peaking at 09:13:
  - dockerA1 trace.to_MG01.duration_p95 — severity ≈ 3.75 (max 4.6142 / global_p95 1.2316), earliest 09:13
  - MG01 trace.from_dockerA1.duration_p95 — severity ≈ 3.75, earliest 09:13
  - dockerA2 trace.to_MG02.duration_p95 — severity ≈ 3.46, earliest 09:13
  - MG02 trace.from_dockerA2.duration_p95 — severity ≈ 3.46, earliest 09:13
  - MG01 / dockerB2 duration_p95 and several duration_mean groups follow with severity ratios ~3.08–3.33 (all with earliest 09:13).
- (Top-20 table is available in kernel variable; these entries dominate the highest severity ranks and mostly share earliest_anom_ts = 2021-03-25 09:13:00 CST.)

2) Focused view: IG01 and traces involving 'Tomcat01'
- IG01 shows earlier small-to-moderate severity anomalies (09:04–09:10) across several trace groups:
  - IG01 trace.from_root.duration_p95 — severity ≈ 1.02, earliest 09:04
  - IG01 trace.to_Tomcat01.row_count — severity ≈ 1.065 (row_count high vs p5), earliest 09:04
  - IG01 trace.self.row_count and other row_count entries — earliest 09:05 with severity ≈ 1.04–1.08
  - IG01 trace.to_Tomcat01.duration_p95 — a notable duration anomaly at 09:13 with severity ≈ 1.595
- A few Tomcat01-related traces on other components (e.g., MG02) also show anomalies at 09:06 and at 09:13.
- (Focused table with these rows is available in kernel variable and sorted by earliest anomaly time.)

3) Top components by mean severity_ratio (among their anomalous trace groups)
- dockerA1 — mean severity ≈ 1.546; total trace anomaly points = 32
- dockerA2 — mean severity ≈ 1.429; total = 32
- MG01    — mean severity ≈ 1.418; total = 82
- dockerB2 — mean severity ≈ 1.404; total = 34
- MG02    — mean severity ≈ 1.398; total = 81

Key takeaways
- The strongest trace severity spikes are concentrated around 09:13 and involve dockerA ↔ MG communication paths (duration metrics), with severity ratios up to ~3.7.
- IG01 exhibits earlier anomalies (09:04–09:10) across multiple trace groups and also shows a duration anomaly to Tomcat01 at 09:13 (severity ≈ 1.60).
- At component-level, dockerA1/dockerA2 and MG01/MG02 stand out by mean severity across their anomalous traces.

If you want the exact compact tables shown in this analysis, they are available in the kernel as:
- full severity-sorted top-20: df_severity_top20
- focused IG01 / Tomcat01-related rows: df_focus_sorted
- top-5 components by mean severity: df_top5_by_mean_severity

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name  series_count  global_p95  window_count  anomaly_count_duration  anomaly_count_rowcount         earliest_anom_ts  max_in_window  mean_in_window  severity_ratio
0   dockerA1         trace.to_MG01.duration_p95            31    1.231600            31                       2                       0  2021-03-25 09:13:00 CST       4.614200        0.567539        3.746509
1       MG01   trace.from_dockerA1.duration_p95            31    1.231600            31                       2                       0  2021-03-25 09:13:00 CST       4.614200        0.567539        3.746509
2   dockerA2         trace.to_MG02.duration_p95            30    1.028310            30                       2                       0  2021-03-25 09:13:00 CST       3.553900        0.542257        3.456059
3       MG02   trace.from_dockerA2.duration_p95            30    1.028310            30                       2                       0  2021-03-25 09:13:00 CST       3.553900        0.542257        3.456059
4       MG01   trace.from_dockerB2.duration_p95            30    1.016433            30                       2                       0  2021-03-25 09:13:00 CST       3.382800        0.535385        3.328111
5   dockerB2         trace.to_MG01.duration_p95            30    1.016433            30                       2                       0  2021-03-25 09:13:00 CST       3.382800        0.535385        3.328111
6       MG02  trace.from_dockerA2.duration_mean            30    0.191966            30                       2                       0  2021-03-25 09:13:00 CST       0.630400        0.089198        3.283914
7   dockerA2        trace.to_MG02.duration_mean            30    0.191966            30                       2                       0  2021-03-25 09:13:00 CST       0.630400        0.089198        3.283914
8       MG01  trace.from_dockerB2.duration_mean            30    0.128705            30                       2                       0  2021-03-25 09:13:00 CST       0.396342        0.074972        3.079456
9   dockerB2        trace.to_MG01.duration_mean            30    0.128705            30                       2                       0  2021-03-25 09:13:00 CST       0.396342        0.074972        3.079456
10      MG02  trace.from_dockerB1.duration_mean            31    0.222424            31                       2                       0  2021-03-25 09:13:00 CST       0.512000        0.085696        2.301905
11  dockerB1        trace.to_MG02.duration_mean            31    0.222424            31                       2                       0  2021-03-25 09:13:00 CST       0.512000        0.085696        2.301905
12  dockerB1           trace.self.duration_mean            31    0.076617            31                       2                       0  2021-03-25 09:13:00 CST       0.160379        0.045402        2.093270
13      MG01            trace.self.duration_p95            31    2.100375            31                       2                       0  2021-03-25 09:13:00 CST       4.034800        1.221561        1.920990
14  dockerA1       trace.from_MG02.duration_p95            31    1.036925            31                       2                       0  2021-03-25 09:13:00 CST       1.987800        1.060673        1.917014
15      MG02     trace.to_dockerA1.duration_p95            31    1.036925            31                       2                       0  2021-03-25 09:13:00 CST       1.987800        1.060673        1.917014
16      MG01  trace.from_dockerA1.duration_mean            31    0.547549            31                       2                       0  2021-03-25 09:13:00 CST       1.014000        0.128497        1.851889
17  dockerA1        trace.to_MG01.duration_mean            31    0.547549            31                       2                       0  2021-03-25 09:13:00 CST       1.014000        0.128497        1.851889
18  dockerA1        trace.to_MG02.duration_mean            31    0.230232            31                       2                       0  2021-03-25 09:13:00 CST       0.413875        0.083116        1.797647
19      MG02  trace.from_dockerA1.duration_mean            31    0.230232            31                       2                       0  2021-03-25 09:13:00 CST       0.413875        0.083116        1.797647,   cmdb_id                         trace_name  series_count   global_p95  window_count  anomaly_count_duration  anomaly_count_rowcount         earliest_anom_ts  max_in_window  mean_in_window  severity_ratio
0    IG01       trace.from_root.duration_p95             4     1.192518             4                       1                       0  2021-03-25 09:04:00 CST       1.219000        0.828188        1.022207
1    IG01        trace.to_Tomcat01.row_count            31   293.000000            31                       0                       2  2021-03-25 09:04:00 CST     312.000000      226.677419        1.064846
2    IG01        trace.to_Tomcat04.row_count            31   269.500000            31                       0                       2  2021-03-25 09:05:00 CST     273.000000      221.322581        1.012987
3    IG01        trace.to_Tomcat03.row_count            31   280.000000            31                       0                       2  2021-03-25 09:05:00 CST     303.000000      222.000000        1.082143
4    IG01               trace.self.row_count            31  2248.000000            31                       0                       2  2021-03-25 09:05:00 CST    2334.000000     1789.000000        1.038256
5    IG01      trace.from_root.duration_mean             4     0.957575             4                       1                       0  2021-03-25 09:06:00 CST       1.037500        0.453561        1.083466
6    MG02      trace.from_Tomcat01.row_count            31   288.500000            31                       0                       2  2021-03-25 09:06:00 CST     309.000000      228.903226        1.071057
7    IG01          trace.from_root.row_count             4    12.400000             4                       0                       1  2021-03-25 09:06:00 CST      13.000000        6.750000        1.048387
8    IG01     trace.to_Tomcat01.duration_p95            31     3.430550            31                       2                       0  2021-03-25 09:13:00 CST       5.472400        1.968634        1.595196
9    MG02  trace.from_Tomcat01.duration_mean            31     0.505405            31                       2                       0  2021-03-25 09:13:00 CST       0.698529        0.254716        1.382117,     cmdb_id  mean_severity_ratio  total_trace_anomaly_points
0  dockerA1             1.545971                          32
1  dockerA2             1.429123                          32
2      MG01             1.417546                          82
3  dockerB2             1.403658                          34
4      MG02             1.397530                          81)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure df_series_stats and df_window_stats exist; otherwise recompute (reuse previous logic)
if 'df_series_stats' not in globals() or 'df_window_stats' not in globals():
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

    # Full-series stats (global thresholds)
    df_series_stats = (
        df_metric.groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(
            series_count=('value','count'),
            global_p95=('value', lambda x: x.quantile(0.95) if x.notna().any() else np.nan),
            global_p90=('value', lambda x: x.quantile(0.90) if x.notna().any() else np.nan),
            global_p5 =('value', lambda x: x.quantile(0.05) if x.notna().any() else np.nan),
            series_min=('value','min'),
            series_max=('value','max')
        )
    )
    df_series_stats['series_count'] = df_series_stats['series_count'].astype(int)

    # Windowed (incident) stats
    start_ts = 1616634000
    end_ts = 1616635800
    df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
    df_window = pd.merge(df_window, df_series_stats[['cmdb_id','kpi_name','global_p95']], on=['cmdb_id','kpi_name'], how='left')
    df_window['is_anom_p95'] = df_window['value'] > df_window['global_p95']

    df_window_stats = df_window.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        window_count=('value','count'),
        anomaly_count_p95=('is_anom_p95','sum'),
        max_in_window=('value','max'),
        mean_in_window=('value','mean')
    )
    earliest = df_window[df_window['is_anom_p95']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(earliest_anom_ts_p95=('timestamp','min'))
    df_window_stats = pd.merge(df_window_stats, earliest, on=['cmdb_id','kpi_name'], how='left')

# Merge full-series stats with window stats
df_mg_all = pd.merge(
    df_series_stats[['cmdb_id','kpi_name','series_count','global_p95']],
    df_window_stats[['cmdb_id','kpi_name','window_count','anomaly_count_p95','earliest_anom_ts_p95','max_in_window','mean_in_window']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Filter to MG01 and MG02 and only groups with anomaly_count_p95 > 0
targets = ['MG01', 'MG02']
df_mg = df_mg_all[df_mg_all['cmdb_id'].isin(targets)].copy()
df_mg['anomaly_count_p95'] = pd.to_numeric(df_mg['anomaly_count_p95'], errors='coerce').fillna(0).astype(int)
df_mg = df_mg[df_mg['anomaly_count_p95'] > 0].copy()

# Convert earliest epoch to localized string and keep epoch for sorting
def epoch_to_local_str(x):
    if pd.isna(x):
        return None
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

df_mg['earliest_anom_epoch'] = pd.to_numeric(df_mg['earliest_anom_ts_p95'], errors='coerce')
df_mg['earliest_anom_ts_p95_local'] = df_mg['earliest_anom_epoch'].apply(epoch_to_local_str)

# Ensure numeric fields for severity calculation
df_mg['global_p95'] = pd.to_numeric(df_mg['global_p95'], errors='coerce')
df_mg['max_in_window'] = pd.to_numeric(df_mg['max_in_window'], errors='coerce')

# Compute severity_ratio (null if global_p95 == 0 or NaN)
df_mg['severity_ratio'] = np.where(
    df_mg['global_p95'].fillna(0) == 0,
    np.nan,
    df_mg['max_in_window'] / df_mg['global_p95']
)

# Sort by earliest_anom_ts_p95_local ascending (using epoch) then severity_ratio descending
df_mg_sorted = df_mg.sort_values(['earliest_anom_epoch','severity_ratio'], ascending=[True, False], na_position='last')

# Select requested display columns and limit to top 20 rows
display_cols = [
    'cmdb_id','kpi_name','series_count','global_p95',
    'window_count','anomaly_count_p95','earliest_anom_ts_p95_local',
    'max_in_window','mean_in_window','severity_ratio'
]
df_mg_top20 = df_mg_sorted[display_cols].head(20).reset_index(drop=True)

# Aggregated per-component summary for MG01 and MG02
df_mg_agg = (
    df_mg.groupby('cmdb_id', as_index=False)
    .agg(
        distinct_anomalous_kpi_count=('kpi_name','nunique'),
        total_anomaly_points=('anomaly_count_p95','sum'),
        earliest_anom_epoch=('earliest_anom_epoch','min')
    )
)
df_mg_agg['earliest_anomaly_ts_local'] = df_mg_agg['earliest_anom_epoch'].apply(epoch_to_local_str)
df_mg_agg = df_mg_agg[['cmdb_id','distinct_anomalous_kpi_count','total_anomaly_points','earliest_anomaly_ts_local']].reset_index(drop=True)

# Return compact outputs (top anomalous KPI rows for MG01/MG02 and per-component aggregates)
df_mg_top20, df_mg_agg

```
Out[9]:
```
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure required variables exist
for var in ['df_mg_top20', 'df_mg_agg']:
    if var not in globals():
        raise NameError(f"Required variable '{var}' not found in kernel state. Please run the MG01/MG02 anomaly computation first.")

# Prepare aggregates
agg = df_mg_agg.copy()
# Safe lookup for MG01 and MG02 rows
mg01_row = agg[agg['cmdb_id'] == 'MG01']
mg02_row = agg[agg['cmdb_id'] == 'MG02']

mg01_count = int(mg01_row['distinct_anomalous_kpi_count'].iloc[0]) if not mg01_row.empty else 0
mg01_points = int(mg01_row['total_anomaly_points'].iloc[0]) if not mg01_row.empty else 0
mg01_earliest = mg01_row['earliest_anomaly_ts_local'].iloc[0] if not mg01_row.empty else None

mg02_count = int(mg02_row['distinct_anomalous_kpi_count'].iloc[0]) if not mg02_row.empty else 0
mg02_points = int(mg02_row['total_anomaly_points'].iloc[0]) if not mg02_row.empty else 0
mg02_earliest = mg02_row['earliest_anomaly_ts_local'].iloc[0] if not mg02_row.empty else None

total_kpis = mg01_count + mg02_count
total_points = mg01_points + mg02_points
overall_earliest = None
if mg01_earliest and mg02_earliest:
    overall_earliest = mg01_earliest if mg01_earliest <= mg02_earliest else mg02_earliest
else:
    overall_earliest = mg01_earliest or mg02_earliest

# Top example KPI groups by severity_ratio (from df_mg_top20 or df_mg if available)
df_examples_source = df_mg_top20.copy()
if 'df_mg' in globals():
    df_examples_source = df_mg.copy()
# Ensure severity_ratio exists
if 'severity_ratio' not in df_examples_source.columns:
    df_examples_source['severity_ratio'] = pd.to_numeric(df_examples_source['severity_ratio'], errors='coerce')

top_examples = (
    df_examples_source.sort_values('severity_ratio', ascending=False, na_position='last')
    .head(3)[['cmdb_id','kpi_name','severity_ratio','earliest_anom_ts_p95_local','anomaly_count_p95']]
)

examples_text = []
for _, r in top_examples.iterrows():
    examples_text.append(
        f"{r['cmdb_id']} | {r['kpi_name']} — severity_ratio={r['severity_ratio']:.3g}, anomalies={int(r.get('anomaly_count_p95',0))}, earliest={r.get('earliest_anom_ts_p95_local')}"
    )
examples_text = examples_text or ["(none)"]

# Build summary string
summary = (
    "MG01 / MG02 metric anomaly summary (incident window 2021-03-25 09:00–09:30 CST):\n\n"
    f"- MG01: {mg01_count} distinct anomalous KPIs, {mg01_points} total anomaly points; earliest anomaly at {mg01_earliest}\n"
    f"- MG02: {mg02_count} distinct anomalous KPIs, {mg02_points} total anomaly points; earliest anomaly at {mg02_earliest}\n\n"
    f"- Combined: {total_kpis} distinct anomalous KPI groups, {total_points} total anomaly points; earliest overall at {overall_earliest}\n\n"
    "- Examples (top 3 by severity_ratio):\n  " + "\n  ".join(examples_text) + "\n\n"
    "Notes: severity_ratio = max_in_window / global_p95 (null if global_p95 == 0). "
    "Detailed per-KPI rows (up to 20) are in kernel variable 'df_mg_top20'; per-component aggregates are in 'df_mg_agg'."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                           kpi_name  series_count    global_p95  window_count  anomaly_count_p95 earliest_anom_ts_p95_local  max_in_window  mean_in_window  severity_ratio
0     MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...             9  1.260000e+01             9                  1    2021-03-25 09:00:00 CST   1.300000e+01    8.888889e+00        1.031746
1     MG01                        OSLinux-CPU_CPU_CPUidleutil            30  7.211060e+01            30                  2    2021-03-25 09:00:00 CST   7.216259e+01    7.169762e+01        1.000721
2     MG02           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem            18  4.738000e+02            18                  1    2021-03-25 09:01:00 CST   5.010000e+02    2.172778e+02        1.057408
3     MG01           OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem            18  1.307500e+02            18                  1    2021-03-25 09:01:00 CST   1.350000e+02    1.229444e+02        1.032505
4     MG02                         OSLinux-CPU_CPU_CPUSysTime            30  1.146849e+00            30                  2    2021-03-25 09:01:00 CST   1.165550e+00    1.011777e+00        1.016306
5     MG02  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...            10  3.379075e+04            10                  1    2021-03-25 09:01:00 CST   3.382000e+04    3.334270e+04        1.000866
6     MG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTps            15  1.005000e+00            15                  1    2021-03-25 09:03:00 CST   1.016667e+00    7.344422e-01        1.011609
7     MG02                    OSLinux-CPU_CPU-1_SingleCpuidle            15  9.711095e+01            15                  1    2021-03-25 09:03:00 CST   9.786915e+01    9.570655e+01        1.007808
8     MG02                    OSLinux-CPU_CPU-3_SingleCpuidle            15  9.573303e+01            15                  1    2021-03-25 09:03:00 CST   9.626977e+01    2.218165e+01        1.005607
9     MG01  OSLinux-OSLinux_FILESYSTEM_-tomcat_FSAvailable...            11  3.370050e+04            11                  1    2021-03-25 09:03:00 CST   3.373300e+04    3.325836e+04        1.000964
10    MG01                    OSLinux-CPU_CPU-1_SingleCpuUtil            10  6.049005e+00            10                  1    2021-03-25 09:04:00 CST   6.648900e+00    4.238740e+00        1.099173
11    MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps            15  3.000000e-01            15                  1    2021-03-25 09:05:00 CST   1.000000e+00    6.666667e-02        3.333333
12    MG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps            14  6.150000e+00            14                  1    2021-03-25 09:05:00 CST   1.200000e+01    1.285714e+00        1.951220
13    MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31  5.000000e-01            31                  2    2021-03-25 09:06:00 CST   1.000000e+00    6.451613e-02        2.000000
14    MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed            31  9.549292e+08            31                  2    2021-03-25 09:06:00 CST   9.698863e+08    5.820550e+08        1.015663
15    MG02                        OSLinux-CPU_CPU_CPUidleutil            30  7.226226e+01            30                  2    2021-03-25 09:06:00 CST   7.232024e+01    7.184840e+01        1.000802
16    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...            15  4.100000e-01            15                  1    2021-03-25 09:07:00 CST   9.000000e-01    1.133333e-01        2.195122
17    MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed            31  9.880494e+08            31                  2    2021-03-25 09:07:00 CST   1.008732e+09    6.470318e+08        1.020933
18    MG01         JVM-Memory_7779_JVM_Memory_HeapMemoryUsage            20  4.814863e+01            20                  1    2021-03-25 09:07:00 CST   4.874390e+01    3.131271e+01        1.012363
19    MG02         JVM-Memory_7779_JVM_Memory_HeapMemoryUsage            21  4.671010e+01            21                  1    2021-03-25 09:07:00 CST   4.690420e+01    2.954275e+01        1.004155,   cmdb_id  distinct_anomalous_kpi_count  total_anomaly_points earliest_anomaly_ts_local
0    MG01                            50                    58   2021-03-25 09:00:00 CST
1    MG02                            44                    52   2021-03-25 09:00:00 CST)```
```